# BÁO CÁO THỰC NGHIỆM ĐỀ TÀI TIỂU LUẬN CUỐI KHÓA
## Học phần: Trí tuệ nhân tạo cho IoT (261AIOT331185_01CLC)
### Đề tài: SINH ĐẶC TRƯNG LOG-MEL CỦA CHỮ SỐ NÓI BẰNG cVAE VÀ ĐÁNH GIÁ THEO GIAO THỨC TSTR
---
- **Mã đề tài:** G1 (Phân nhóm: Tạo sinh)
- **Sinh viên thực hiện:** Đặng Quốc Thành Tài
- **Mã số sinh viên:** 23110149
- **Giảng viên hướng dẫn:** ThS/TS. Hồ Nhựt Minh
- **Bộ dữ liệu:** Free Spoken Digit Dataset (FSDD - 3.000 file WAV)

### 1. Thiết lập môi trường và cấu hình tham số

In [ ]:
import sys
from pathlib import Path

# Thêm thư mục gốc vào sys.path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from src.config import (
    DATA_DIR, MANIFEST_PATH, NORM_STATS_PATH, TABLES_DIR, FIGURES_DIR, CHECKPOINT_DIR, EXPORT_DIR,
    AudioConfig, CVAEConfig, ClassifierConfig, ExperimentConfig
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Thư mục dữ liệu: {DATA_DIR}")

### 2. Kiểm tra tính toàn vẹn của Dữ liệu & Phân chia không rò rỉ

In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)
print(f"Tổng số mẫu trong Manifest: {len(manifest_df)}")

# Thống kê theo từng tập
split_summary = manifest_df.groupby(["split", "digit"]).size().unstack(fill_value=0)
print("\n--- Phân bổ số mẫu theo chữ số và tập dữ liệu ---")
display(split_summary)

# Kiểm tra giao thoa giữa các tập
train_files = set(manifest_df[manifest_df["split"] == "train"]["file_id"])
val_files = set(manifest_df[manifest_df["split"] == "val"]["file_id"])
test_files = set(manifest_df[manifest_df["split"] == "test"]["file_id"])

print(f"\nSố file trùng Train & Val: {len(train_files.intersection(val_files))}")
print(f"Số file trùng Train & Test: {len(train_files.intersection(test_files))}")
print(f"Số file trùng Val & Test: {len(val_files.intersection(test_files))}")
print("-> KẾT QUẢ: 100% không rò rỉ giữa các tập dữ liệu!")

### 3. Thống kê Chuẩn hóa z-score (Tính nghiêm ngặt trên tập Train)

In [ ]:
with open(NORM_STATS_PATH, "r", encoding="utf-8") as f:
    norm_stats = json.load(f)

mu = np.array(norm_stats["mu"])
sigma = np.array(norm_stats["sigma"])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(mu, color='navy', lw=2)
ax[0].set_title("Trung bình $\\mu$ trên 64 dải Mel (Train)")
ax[0].set_xlabel("Dải tần Mel (0-63)")
ax[0].set_ylabel("dB")
ax[0].grid(True, alpha=0.3)

ax[1].plot(sigma, color='darkorange', lw=2)
ax[1].set_title("Độ lệch chuẩn $\\sigma$ trên 64 dải Mel (Train)")
ax[1].set_xlabel("Dải tần Mel (0-63)")
ax[1].set_ylabel("dB")
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Tỷ lệ file train dài hơn 8.000 mẫu bị cắt: {norm_stats['trimmed_count']}/{norm_stats['total_train_samples']} ({norm_stats['trimmed_count']/norm_stats['total_train_samples']*100:.2f}%)")

### 4. Kết quả Thực nghiệm Chính: So sánh ghép cặp TRTR vs TSTR (3 Seed)

In [ ]:
summary_csv = TABLES_DIR / "trtr_vs_tstr_summary.csv"
if summary_csv.exists():
    res_df = pd.read_csv(summary_csv)
    display(res_df)
else:
    print("Chưa tìm thấy bảng kết quả tổng hợp. Hãy chạy pipeline trước.")

### 5. Trực quan hóa Ma trận Nhầm lẫn: TRTR vs TSTR (Seed 42)

In [ ]:
cm_img_path = FIGURES_DIR / "confusion_matrix_trtr_vs_tstr_seed42.png"
if cm_img_path.exists():
    from IPython.display import Image
    display(Image(filename=str(cm_img_path)))
else:
    print(f"Chưa có hình: {cm_img_path}")

### 6. Trực quan hóa Phổ Log-Mel: Thật vs Tái tạo vs Sinh từ prior $z \sim \mathcal{N}(0, I)$

In [ ]:
from IPython.display import Image
for digit in [0, 1, 5, 7]:
    p = FIGURES_DIR / f"spectrogram_comparison_digit_{digit}.png"
    if p.exists():
        print(f"--- Phổ Log-Mel Chữ số {digit} ---")
        display(Image(filename=str(p)))

### 7. Khảo sát độ nhạy Trọng số $\beta \in \{0.0, 0.1, 1.0\}$ (Regularization Ablation)

In [ ]:
beta_csv = TABLES_DIR / "ablation_beta_summary.csv"
if beta_csv.exists():
    beta_df = pd.read_csv(beta_csv)
    display(beta_df)
else:
    print("Chưa có bảng khảo sát beta.")

### 8. Đo lường Chi phí Tính toán (Computational Benchmark)

In [ ]:
bench_csv = TABLES_DIR / "computational_cost_benchmark.csv"
if bench_csv.exists():
    bench_df = pd.read_csv(bench_csv)
    display(bench_df)
else:
    print("Chưa có bảng benchmark.")

### 9. Đóng gói Xuất Mô hình cho Vi Điều Khiển ESP32 (TinyML)

In [ ]:
esp32_json = EXPORT_DIR / "esp32_memory_feasibility.json"
if esp32_json.exists():
    with open(esp32_json, "r", encoding="utf-8") as f:
        esp32_info = json.load(f)
    print(json.dumps(esp32_info, indent=2, ensure_ascii=False))
    
header_file = EXPORT_DIR / "model_data.h"
if header_file.exists():
    print(f"\n-> File C Header cho ESP32 đã sẵn sàng: {header_file} ({header_file.stat().st_size / 1024.0:.2f} KB)")

### 10. Nghe thử Âm thanh Khôi phục xấp xỉ bằng Griffin-Lim

In [ ]:
from IPython.display import Audio
audio_dir = OUTPUT_DIR / "audio_reconstructed"
for wav in audio_dir.glob("*.wav"):
    print(f"Nghe thử file: {wav.name}")
    display(Audio(filename=str(wav)))